# Prompt Workbench

Scratchpad for experimenting with agent prompts and tools **before** committing them to `src/ifta/agent/`.

Workflow:
1. Tweak the prompt below.
2. Run the cell.
3. Read the response, iterate.
4. When it's good, copy into `src/ifta/agent/prompts.py` or `tools.py`.

Choose the kernel **IFTA Pipeline (.venv)** in the top-right when this notebook opens.

In [ ]:
from ifta.agent import ask, review, run_agent
from ifta.agent.tools import ALL_TOOLS
from ifta.agent.prompts import SYSTEM_PROMPT

print(f'{len(ALL_TOOLS)} tools loaded')
for t in ALL_TOOLS:
    print(f'  - {t.name}')

## Quick ask (Haiku — cheap)

Use Haiku for fast experimentation. Switch to Opus once the prompt is working.

In [ ]:
answer = ask(
    'What is David\'s base state and current fleet size?',
    model='claude-haiku-4-5',
    max_tokens=1024,
)
print(answer)

## Try a custom system prompt

Bypass the built-in system prompt to test a variation:

In [ ]:
import anthropic
from ifta.agent.tools import ALL_TOOLS

client = anthropic.Anthropic()

EXPERIMENTAL_SYSTEM = '''\
You are a CFO-style IFTA reviewer. Speak in clipped bullets — no preamble.
Focus on cost optimization opportunities and audit risk.'''

runner = client.beta.messages.tool_runner(
    model='claude-haiku-4-5',
    max_tokens=1024,
    system=EXPERIMENTAL_SYSTEM,
    tools=ALL_TOOLS,
    messages=[{'role': 'user', 'content': 'Review Q4-2025 from a CFO perspective.'}],
)

for msg in runner:
    for block in msg.content:
        if block.type == 'text':
            print(block.text)

## Inspect a tool call directly

Call a tool function directly (without going through the model) to see what the agent sees:

In [ ]:
from ifta.agent.tools import query_return, get_david_profile
import json

# Tools are @beta_tool decorated — call .func to invoke the underlying function
result = query_return.func(quarter='Q4-2025')
data = json.loads(result)
print(f'Fleet MPG: {data["fleet_mpg"]}')
print(f'Total tax due: ${data["total_tax_due"]}')
print(f'Jurisdiction lines: {len(data["lines"])}')

## Count tokens before sending

Useful when designing prompts — see what the system prompt + a sample question will cost.

In [ ]:
import anthropic

client = anthropic.Anthropic()

count = client.messages.count_tokens(
    model='claude-opus-4-7',
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': 'Review Q4-2025 for me.'}],
)
print(f'Input tokens: {count.input_tokens}')
print(f'Estimated cost (input only): ${count.input_tokens / 1_000_000 * 5:.4f}')